In [3]:
class Account:

    bank_name = "State Bank of India (SBI)"
    minimum_balance = 1000  # class-level minimum balance rule

    def __init__(self, name, bal, acc, pin):
        self.name = name
        self.balance = bal
        self.account_no = acc
        self.__pin = pin          # private attribute
        self.is_active = True
        self.transaction_history = []

    # ─────────────────────────────────────────
    #  Internal helpers
    # ─────────────────────────────────────────

    def __verify_pin(self, pin):
        return self.__pin == pin

    def __log_transaction(self, t_type, amount):
        entry = f"{t_type:10} | Rs {amount:<10} | Balance: Rs {self.balance}"
        self.transaction_history.append(entry)

    def __check_active(self):
        if not self.is_active:
            print("\n    Account is closed. No transactions allowed.")
            return False
        return True

    # ─────────────────────────────────────────
    #  Debit / Withdrawal
    # ─────────────────────────────────────────

    def debit(self, amount, pin):
        print("\n" + "─" * 45)
        print(f"    WITHDRAWAL REQUEST  |  A/C: {self.account_no}")
        print("─" * 45)

        if not self.__check_active():
            return
        if not self.__verify_pin(pin):
            print("    Incorrect PIN. Transaction cancelled.")
            return
        if amount <= 0:
            print("    Invalid amount. Enter a value greater than 0.")
            return
        if amount > self.balance:
            print("    Insufficient funds.")
            print(f"      Available balance: Rs {self.balance}")
            return
        if (self.balance - amount) < Account.minimum_balance:
            print(f"    Cannot withdraw. Minimum balance of Rs {Account.minimum_balance} must be maintained.")
            return
        if amount % 100 != 0:
            print("    Amount must be in multiples of Rs 100.")
            return

        self.balance -= amount
        self.__log_transaction("DEBIT", amount)
        print(f"    Rs {amount} debited successfully.")
        print(f"    Remaining Balance : Rs {self.balance}")
        print("─" * 45)

    # ─────────────────────────────────────────
    #  Credit / Deposit
    # ─────────────────────────────────────────

    def credit(self, amount):
        print("\n" + "─" * 45)
        print(f"   DEPOSIT REQUEST  |  A/C: {self.account_no}")
        print("─" * 45)

        if not self.__check_active():
            return
        if amount <= 0:
            print("    Invalid amount. Enter a value greater than 0.")
            return
        if amount > 100000:
            print("    Cash deposit above Rs 1,00,000 requires a manager's approval.")
            return

        self.balance += amount
        self.__log_transaction("CREDIT", amount)
        print(f"    Rs {amount} credited successfully.")
        print(f"    Updated Balance   : Rs {self.balance}")
        print("─" * 45)

    # ─────────────────────────────────────────
    #  Transfer
    # ─────────────────────────────────────────

    def transfer(self, amount, pin, target_account):
        print("\n" + "─" * 45)
        print(f"    TRANSFER REQUEST  |  A/C: {self.account_no}")
        print("─" * 45)

        if not self.__check_active():
            return
        if not self.__verify_pin(pin):
            print("    Incorrect PIN. Transaction cancelled.")
            return
        if amount <= 0:
            print("   Invalid amount.")
            return
        if amount > self.balance:
            print("    Insufficient funds for transfer.")
            return
        if (self.balance - amount) < Account.minimum_balance:
            print(f"    Transfer would breach the minimum balance of Rs {Account.minimum_balance}.")
            return
        if not target_account.is_active:
            print("    Recipient's account is closed. Transfer failed.")
            return

        self.balance -= amount
        target_account.balance += amount
        self.__log_transaction("TRANSFER OUT", amount)
        target_account.__log_transaction("TRANSFER IN", amount)
        print(f"    Rs {amount} transferred to A/C {target_account.account_no} ({target_account.name}).")
        print(f"    Your Remaining Balance : Rs {self.balance}")
        print("─" * 45)

    # ─────────────────────────────────────────
    #  Balance Enquiry
    # ─────────────────────────────────────────

    def get_balance(self, pin):
        print("\n" + "─" * 45)
        print(f"    BALANCE ENQUIRY  |  A/C: {self.account_no}")
        print("─" * 45)

        if not self.__check_active():
            return
        if not self.__verify_pin(pin):
            print("    Incorrect PIN.")
            return

        print(f"  Account Holder : {self.name}")
        print(f"  Account No.    : {self.account_no}")
        print(f"  Bank           : {Account.bank_name}")
        print(f"  Balance        : Rs {self.balance}")
        print("─" * 45)

    # ─────────────────────────────────────────
    #  Transaction History
    # ─────────────────────────────────────────

    def show_history(self, pin):
        print("\n" + "─" * 45)
        print(f"    TRANSACTION HISTORY  |  A/C: {self.account_no}")
        print("─" * 45)

        if not self.__check_active():
            return
        if not self.__verify_pin(pin):
            print("   Incorrect PIN.")
            return
        if not self.transaction_history:
            print("  No transactions found.")
        else:
            for i, record in enumerate(self.transaction_history, 1):
                print(f"  {i}. {record}")
        print("─" * 45)

    # ─────────────────────────────────────────
    #  Close Account
    # ─────────────────────────────────────────

    def close_account(self, pin):
        print("\n" + "─" * 45)
        print(f"    CLOSE ACCOUNT  |  A/C: {self.account_no}")
        print("─" * 45)

        if not self.__verify_pin(pin):
            print("    Incorrect PIN. Cannot close account.")
            return

        self.is_active = False
        print(f"   Account {self.account_no} has been closed.")
        print(f"   Final Balance paid out: Rs {self.balance}")
        self.balance = 0
        print("─" * 45)


# ═══════════════════════════════════════════════
#  Demo
# ═══════════════════════════════════════════════

print("\n" + "═" * 45)
print(f"   Welcome to {Account.bank_name}")
print("═" * 45)

acc1 = Account("Ravi Kumar",  10000, 112233, 1111)
acc2 = Account("Priya Sharma", 5000, 445566, 2222)

# Balance check
acc1.get_balance(1111)

# Wrong PIN
acc1.get_balance(9999)

# Valid debit
acc1.debit(2000, 1111)

# Debit that breaks minimum balance
acc1.debit(8000, 1111)

# Credit
acc1.credit(3000)

# Deposit above limit
acc1.credit(200000)

# Transfer
acc1.transfer(2000, 1111, acc2)

# Transaction history
acc1.show_history(1111)

# Close account
acc2.close_account(2222)

# Try transacting on closed account
acc2.credit(500)



═════════════════════════════════════════════
   Welcome to State Bank of India (SBI)
═════════════════════════════════════════════

─────────────────────────────────────────────
    BALANCE ENQUIRY  |  A/C: 112233
─────────────────────────────────────────────
  Account Holder : Ravi Kumar
  Account No.    : 112233
  Bank           : State Bank of India (SBI)
  Balance        : Rs 10000
─────────────────────────────────────────────

─────────────────────────────────────────────
    BALANCE ENQUIRY  |  A/C: 112233
─────────────────────────────────────────────
    Incorrect PIN.

─────────────────────────────────────────────
    WITHDRAWAL REQUEST  |  A/C: 112233
─────────────────────────────────────────────
    Rs 2000 debited successfully.
    Remaining Balance : Rs 8000
─────────────────────────────────────────────

─────────────────────────────────────────────
    WITHDRAWAL REQUEST  |  A/C: 112233
─────────────────────────────────────────────
    Cannot withdraw. Minimum balance of